# VERITAS 02 — Transformer from scratch

**Phase 2 of 16.** Decoder-only, pre-norm, RoPE + GQA + SwiGLU + RMSNorm.

## Attention, derived

For one head with `Q, K, V ∈ ℝ^{L×d}`:

$$ A = \mathrm{softmax}\!\left(\frac{QK^\top}{\sqrt{d}} + M\right), \qquad O = AV $$

**Why `1/√d`?** If `q,k` have i.i.d. unit-variance entries, `q·k = Σᵢ qᵢkᵢ` has
variance `d`. Unscaled, logits grow like `√d`, softmax saturates, and its
Jacobian `diag(a) − aaᵀ` → 0: the gradient vanishes. Dividing by `√d` pins the
logit variance at ~1 for any head size.

**Why the causal mask?** An LM factorises `p(x₁..x_L) = Π p(x_t | x_<t)`.
Setting `M_ij = −∞` for `j > i` gives every position exactly its past, so all
`L` positions train in parallel from one forward pass.

**Why multi-head?** One softmax = one convex combination of value vectors = one
lookup. `h` heads of width `d/h` cost identical FLOPs and give `h` independent
lookups.

## Four architecture choices, and the reason for each

| choice | alternative | why this one |
|---|---|---|
| **RoPE** | learned absolute pos. | rotations are orthogonal ⇒ `⟨R_m q, R_n k⟩ = ⟨q, R_{n−m} k⟩`: logits depend only on *relative* distance. Evidence chunks concatenate in any order; context extrapolates; zero position parameters. |
| **GQA** (`n_kv < n_heads`) | full MHA | KV cache = `2·L·n_kv·d_head·n_layers·2 B`. 8 heads → 2 KV heads is a **4× smaller cache**, which is what fits a long evidence context on one GPU. |
| **RMSNorm** | LayerNorm | drops mean-subtraction and bias: ~2× fewer reduction passes, fewer params, equal quality. |
| **SwiGLU** | GELU FFN | a gate `W₃x` multiplies the activation — a data-dependent interaction a single-matrix FFN cannot express. Width `8/3·d` keeps params equal to a `4d` GELU FFN. |

Plus **weight tying** (embedding = output head: inverse maps over the same
vocabulary, saves `V·d` params) and **residual-scaled init** (`1/√(2·n_layers)`
on `wo`/`w2`, or the residual stream's variance grows linearly with depth).

In [1]:
import sys, os, time, math, json, random
from pathlib import Path
ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
sys.path.insert(0, str(ROOT))
import numpy as np, torch
torch.manual_seed(1337); np.random.seed(1337); random.seed(1337)
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print('device:', DEVICE, '| torch', torch.__version__)
if DEVICE == 'cuda':
    print('gpu:', torch.cuda.get_device_name(0),
          f'| {torch.cuda.get_device_properties(0).total_memory/1e9:.1f} GB')

device: cuda | torch 2.11.0+cu128
gpu: NVIDIA GeForce RTX 5060 Laptop GPU | 8.5 GB


In [3]:
from veritas.model.attention import build_rope_cache, apply_rope, naive_attention, KVCache
from veritas.model.transformer import ModelConfig, VeritasLM, RMSNorm, SwiGLU
import torch.nn.functional as F

## Check 1 — the fused kernel computes the textbook equation

`F.scaled_dot_product_attention` dispatches to FlashAttention: tiled online
softmax, so the `L×L` matrix is **never materialised** (memory `O(L²) → O(L)`,
2–4× faster). It must agree with the explicit loop to within float error.

In [4]:
q,k,v = [torch.randn(2, 4, 32, 16) for _ in range(3)]
ref  = naive_attention(q, k, v, causal=True)
fast = F.scaled_dot_product_attention(q, k, v, is_causal=True)
print('max abs diff:', (ref-fast).abs().max().item())
assert torch.allclose(ref, fast, atol=1e-5)
print('PASS — the fused kernel is the same function, computed without the L×L matrix')

max abs diff: 3.5762786865234375e-07
PASS — the fused kernel is the same function, computed without the L×L matrix


## Check 2 — RoPE really is relative

Rotate a query at position `m` and a key at position `n`. The attention logit
must depend only on `n − m`. This is the property that lets the model handle
evidence pasted in at arbitrary offsets.

In [5]:
cos, sin = build_rope_cache(64, 16)
qv, kv_ = torch.randn(1,1,1,16), torch.randn(1,1,1,16)
def logit(m, n):
    qq = apply_rope(qv, cos[m:m+1], sin[m:m+1])
    kk = apply_rope(kv_, cos[n:n+1], sin[n:n+1])
    return (qq*kk).sum().item()
print(f'(m=2,n=5)  d=3 -> {logit(2,5):+.6f}')
print(f'(m=10,n=13) d=3 -> {logit(10,13):+.6f}')
print(f'(m=30,n=33) d=3 -> {logit(30,33):+.6f}')
print(f'\n(m=2,n=9)  d=7 -> {logit(2,9):+.6f}   <- different distance, different logit')
assert abs(logit(2,5)-logit(30,33)) < 1e-4
print('\nPASS — same offset gives the same logit anywhere in the sequence')

(m=2,n=5)  d=3 -> +4.774406
(m=10,n=13) d=3 -> +4.774406
(m=30,n=33) d=3 -> +4.774407

(m=2,n=9)  d=7 -> +4.537181   <- different distance, different logit

PASS — same offset gives the same logit anywhere in the sequence


## Check 3 — initial loss must be ≈ ln(V)

At initialisation the model knows nothing, so it should predict uniform over
the vocabulary: loss `= −ln(1/V) = ln V`. A materially lower value at step 0
means label leakage; a much higher value means a broken initialisation. This is
the cheapest bug-catcher in all of LM training — run it before every training
job.

In [6]:
tok_path = ROOT/'checkpoints'/'tokenizer.json'
from veritas.tokenizer.bpe import BPETokenizer
if tok_path.exists():
    tok = BPETokenizer.load(tok_path); V = tok.vocab_size
else:
    V = 4096; tok = None
    print('run notebook 01 first for a real tokenizer; using V=4096 for the shape checks')

cfg = ModelConfig(vocab_size=V, d_model=384, n_layers=8, n_heads=8, n_kv_heads=2, max_seq_len=512)
model = VeritasLM(cfg)
print(f'params: {model.num_params():,} total | {model.num_params(True):,} non-embedding')
print(f'd_ff (SwiGLU, 8/3·d rounded to 64): {cfg.d_ff}')

x = torch.randint(0, V, (4, 128)); y = torch.randint(0, V, (4, 128))
_, loss = model(x, y)
print(f'\ninit loss {loss.item():.4f} vs ln(V) = {math.log(V):.4f}')
assert abs(loss.item() - math.log(V)) < 0.25
print('PASS')

params: 12,632,448 total | 12,392,832 non-embedding
d_ff (SwiGLU, 8/3·d rounded to 64): 1024

init loss 6.5253 vs ln(V) = 6.4362
PASS


## Check 4 — the KV cache does not change the output

Without a cache, generating `T` tokens re-encodes the prefix every step:
`O(T³)` total. With it, each step is `O(T)` → `O(T²)` overall. It is pure
memoisation, so **greedy output must be bit-identical** with and without it.
Any mismatch is a position-indexing bug (usually the RoPE offset).

In [7]:
model.eval()
prompt = torch.randint(0, V, (2, 8))
with torch.inference_mode():
    t0=time.time(); a = model.generate(prompt, max_new_tokens=48, temperature=0.0, use_cache=True);  t_cache=time.time()-t0
    t0=time.time(); b = model.generate(prompt, max_new_tokens=48, temperature=0.0, use_cache=False); t_nocache=time.time()-t0
print('identical output:', torch.equal(a,b))
print(f'with cache {t_cache:.3f}s | without {t_nocache:.3f}s | speedup {t_nocache/t_cache:.1f}x')
assert torch.equal(a,b)
print('PASS')

identical output: True
with cache 0.789s | without 2.520s | speedup 3.2x
PASS


## Check 5 — GQA memory saving, measured

The KV cache dominates inference memory at long context. This is the number to
quote when asked why GQA is in the architecture.

In [8]:
def kv_bytes(cfg, seq, batch=1, dtype_bytes=2):
    return 2*cfg.n_layers*batch*cfg.n_kv_heads*(cfg.d_model//cfg.n_heads)*seq*dtype_bytes
mha = ModelConfig(vocab_size=V, d_model=384, n_layers=8, n_heads=8, n_kv_heads=8, max_seq_len=512)
for seq in (512, 2048, 8192):
    a, b = kv_bytes(mha, seq), kv_bytes(cfg, seq)
    print(f'seq {seq:5d}: MHA {a/1e6:7.1f} MB | GQA(2 kv) {b/1e6:6.1f} MB | {a/b:.0f}x smaller')

seq   512: MHA     6.3 MB | GQA(2 kv)    1.6 MB | 4x smaller
seq  2048: MHA    25.2 MB | GQA(2 kv)    6.3 MB | 4x smaller
seq  8192: MHA   100.7 MB | GQA(2 kv)   25.2 MB | 4x smaller


## Check 6 — gradients reach layer 0

Pre-norm leaves an unnormalised identity path from the loss to the embedding.
Print the gradient norm per layer: it should be the same order of magnitude
everywhere. A norm collapsing toward zero at layer 0 means the residual path is
broken (the classic post-norm deep-stack failure).

In [9]:
model.train(); model.zero_grad()
_, loss = model(x, y); loss.backward()
norms = [ (n, p.grad.norm().item()) for n,p in model.named_parameters()
          if p.grad is not None and 'attn.wq.weight' in n ]
for n, g in norms: print(f'{n:42s} grad-norm {g:.5f}')
vals = [g for _, g in norms]
print(f'\nratio deepest/shallowest = {max(vals)/max(min(vals),1e-12):.2f}  (want O(1), not 10^k)')

blocks.0.attn.wq.weight                    grad-norm 0.05451
blocks.1.attn.wq.weight                    grad-norm 0.02400
blocks.2.attn.wq.weight                    grad-norm 0.01633
blocks.3.attn.wq.weight                    grad-norm 0.01148
blocks.4.attn.wq.weight                    grad-norm 0.01039
blocks.5.attn.wq.weight                    grad-norm 0.00931
blocks.6.attn.wq.weight                    grad-norm 0.00691
blocks.7.attn.wq.weight                    grad-norm 0.00697

ratio deepest/shallowest = 7.89  (want O(1), not 10^k)


## Parameter budget

Where the parameters actually go, so you can size the model for your GPU.

In [10]:
def budget(cfg):
    d, V_, L = cfg.d_model, cfg.vocab_size, cfg.n_layers
    hd = d//cfg.n_heads
    emb  = V_*d
    attn = L*(d*cfg.n_heads*hd + 2*d*cfg.n_kv_heads*hd + cfg.n_heads*hd*d)
    ffn  = L*3*d*cfg.d_ff
    norm = L*2*d + d
    return {'embedding (tied w/ head)': emb, 'attention': attn, 'ffn (SwiGLU)': ffn, 'norms': norm}
bud = budget(cfg); tot = sum(bud.values())
for k_, v_ in bud.items(): print(f'{k_:26s} {v_:>11,}  ({100*v_/tot:4.1f}%)')
print(f'{"TOTAL":26s} {tot:>11,}')
print(f'\nfp32 weights {tot*4/1e6:.0f} MB | AdamW states (m,v) {tot*8/1e6:.0f} MB'
      f' | ≈{tot*16/1e6:.0f} MB before activations')

embedding (tied w/ head)       239,616  ( 1.9%)
attention                    2,949,120  (23.3%)
ffn (SwiGLU)                 9,437,184  (74.7%)
norms                            6,528  ( 0.1%)
TOTAL                       12,632,448

fp32 weights 51 MB | AdamW states (m,v) 101 MB | ≈202 MB before activations


Next: **03 — pretraining.**